## **Ejercicio 1**

In [ ]:
import pandas as pd
df = pd.read_csv('info_customers.csv')
df.head()

In [ ]:
df.isna().sum()

### Analisis Exploratorio


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.countplot(data=df, x='Churn') # Importante para ver el desbalance de clases
plt.title("Distribución de Churn") 
plt.show()

In [ ]:
sns.boxplot(data=df, x="Churn", y="tenure")
plt.title("Tenure según Churn")
plt.show()

In [ ]:
sns.countplot(data=df, x="Contract", hue="Churn")
plt.xticks(rotation=45)
plt.title("Churn según tipo de contrato")
plt.show()

In [ ]:
# Histograma
sns.histplot(x = "MonthlyCharges", hue = "Churn", data = df)

plt.show()

### Algunas columnas no son reconocidas como numericas


In [ ]:
# Veo el tipo de dato
df.dtypes

In [ ]:
df.nunique()

In [ ]:
for col in df.columns:
    if col != 'customerID':
        print(f"\nColumna: {col}")
        print(df[col].value_counts())

In [ ]:
df = pd.get_dummies(df, columns=["gender", "Partner", "Dependents", "Contract", "Churn", "PaymentMethod", "PaperlessBilling", "StreamingMovies", "StreamingTV", "TechSupport", "DeviceProtection", "OnlineBackup", "OnlineSecurity", "InternetService", "MultipleLines", "PhoneService"])
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

*Borro Las columnas customerID y Churn_No, ya que no tiene sentido usarlas. customerID solo identifica al cliente y mi variable target tiene mas sentido que sea Churn_Yes*

In [ ]:
df = df.drop(columns=["customerID"])
df = df.drop(columns=["Churn_No"])

*Ahora todas las columnas son numericas.*

In [ ]:
df.dtypes

 ## **Ejercicio 2**

*Antes de resolver el problema de los valores nulos, tengo que separar la muestra.*
*Si no, estaria haciendo Data Leakage. Porque estaria usando informacion del test para el training.*

## **Valores Faltantes**

*Se reemplazan los valores faltantes por la media para mantener un valor representativo sin alterar significativamente la distribución de los datos.*

In [ ]:
# Separo Training y Test
from sklearn.model_selection import train_test_split

X = df.drop("Churn_Yes", axis=1)
y = df["Churn_Yes"]

# Training (80%), Test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="mean")
imputer.fit(X_train)
X_train = imputer.transform(X_train)
X_test = imputer.transform(X_test)

## **Escalado**

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


## **Cross-Validation con K-Fold (K=5)**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# K-FOLD
kf = KFold(n_splits=5, shuffle=True, random_state=42)

accuracies = []
precisions = []
recalls = []
f1s = []

X_train_df = pd.DataFrame(X_train)
for train_index, val_index in kf.split(X_train):
    
    # separar datos
    X_train_fold = X_train_df.iloc[train_index].copy()
    X_val_fold = X_train_df.iloc[val_index].copy()
    y_train_fold = y.iloc[train_index]
    y_val_fold = y.iloc[val_index]

    # modelo
    modelo = LogisticRegression(max_iter=1000)
    modelo.fit(X_train_fold, y_train_fold)

    # predicción
    y_pred = modelo.predict(X_val_fold)

    # métricas
    acc = accuracy_score(y_val_fold, y_pred)
    prec = precision_score(y_val_fold, y_pred)
    rec = recall_score(y_val_fold, y_pred)
    f1 = f1_score(y_val_fold, y_pred)
    cm = confusion_matrix(y_val_fold, y_pred)

    # guardar
    accuracies.append(acc)
    precisions.append(prec)
    recalls.append(rec)
    f1s.append(f1)

    print("Confusion matrix:\n", cm)
    print("------")

# resultados finales
print("Accuracy promedio:", np.mean(accuracies))
print("Precision promedio:", np.mean(precisions))
print("Recall promedio:", np.mean(recalls))
print("F1-score promedio:", np.mean(f1s))

## **Evaluando el modelo sobre todo el training set**

In [ ]:
modelo = LogisticRegression(max_iter=1000) 

modelo.fit(X_train, y_train)

# predicción
y_pred = modelo.predict(X_test)

# métricas
accuracy_Test = accuracy_score(y_test, y_pred)
precision_Test = precision_score(y_test, y_pred)
recall_Test = recall_score(y_test, y_pred)
f1s_Test = f1_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("Confusion matrix:\n", cm)
print("------")

# resultados finales
print("Accuracy promedio:", accuracy_Test)
print("Precision promedio:", precision_Test)
print("Recall promedio:", recall_Test)
print("F1-score promedio:", f1s_Test)


## Balanceando el dataset con Oversampling

In [ ]:
from imblearn.over_sampling import RandomOverSampler
from collections import Counter

rus = RandomOverSampler(random_state=42)

X_train_res, y_train_res = rus.fit_resample(X_train, y_train)

print("Distribución original:", Counter(y_train))
print("Distribución balanceada:", Counter(y_train_res))

In [ ]:
#Entrenar con los datos balanceados
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

model_bal = LogisticRegression()

model_bal.fit(X_train_res, y_train_res)


y_pred_bal = model_bal.predict(X_test)

print(confusion_matrix(y_test, y_pred_bal))
print("\n")
print(classification_report(y_test, y_pred_bal))



## **Parte II: Reformulando el problema**

*La columna que elegimos usar como nuevo target es SeniorCitizen*

*Vamos a intentar predecir si el cliente es un Senior Citizen o no*

In [ ]:
from sklearn.linear_model import LogisticRegression
X = df.drop("SeniorCitizen", axis=1)
y = df["SeniorCitizen"]

# Training (80%), Test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Imputer
imputer = SimpleImputer(strategy="mean")
imputer.fit(X_train)
X_train = imputer.transform(X_train)
X_test = imputer.transform(X_test)

modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_train, y_train)

# predicción
y_pred = modelo.predict(X_test)

# métricas
accuracy_Test = accuracy_score(y_test, y_pred)
precision_Test = precision_score(y_test, y_pred)
recall_Test = recall_score(y_test, y_pred)
f1s_Test = f1_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("Confusion matrix:\n", cm)
print("------")

# resultados finales
print("Accuracy promedio:", accuracy_Test)
print("Precision promedio:", precision_Test)
print("Recall promedio:", recall_Test)
print("F1-score promedio:", f1s_Test)